# 30 · Benchmarks: why Spark + Sedona — Part D, step D10 (schools)

One fixed job, the core of the E2SFCA step for primary schools: every (cell, school) pair within 1 km on the WGS84
ellipsoid, the Gaussian weight, and per school the sum of children × weight. It is timed

1. **by data size:** Kerala 10 % sample → Kerala (2.98 M cells) → Uttar Pradesh (13.75 M cells); the all-India E2SFCA
   (140 M cells, all four levels) took about 5 h in notebook 26 and is not repeated;
2. **by cores:** Spark with 1, 2, 4 and 6 cores (Kerala and Uttar Pradesh);
3. **Spark + Sedona vs plain Python:** Sedona broadcast index join (the D4 winner) against one Python process
   (scipy KD-tree + pyproj ellipsoid distance, cells in chunks of 1 M).

Every setting runs in its own process, 3 times (median reported), with its peak memory recorded; a process over the
memory or time limit is stopped and recorded as such. **Check:** every engine and core count gives the same pair count
and weighted children, and the plain-Python job equals a brute-force search on 2,000 cells.

**Test mode** (default): 10 % Kerala sample only. **Full run:** set `BENCH_FULL=1` before starting (laptop idle).
Outputs (full run): `FINAL COMPLETE DATASET/results/benchmarks/` `benchmarks.csv`, `bench_size.png`,
`bench_cores.png`, `five_v.csv`.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Settings

In [4]:
import json
import os
import shutil
import subprocess
import threading
import time

import duckdb
import numpy as np
import pandas as pd
import psutil

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import population_cells as pc

# FULL run (tonight, laptop idle): set the environment variable BENCH_FULL=1 before starting the notebook.
# Test run (default): the 10 % Kerala sample only, 2 Spark cores, small memory.
FULL = os.environ.get("BENCH_FULL") == "1"
SILVER, GOLD = PROJECT / "data" / "lake" / "silver", PROJECT / "data" / "lake" / "gold"
BENCH = (Path("/content/tmp") if IN_COLAB else PROJECT / "data" / "tmp") / "bench"
OUT = PROJECT / "FINAL COMPLETE DATASET" / "results" / "benchmarks"
JOB = PROJECT / "scripts" / "bigdata" / "bench_job.py"
LOG = BENCH / ("results_full.jsonl" if FULL else "results_test.jsonl")
BENCH.mkdir(parents=True, exist_ok=True)
if not FULL and LOG.exists():
    LOG.unlink()  # a test run starts clean; the full run keeps finished settings (restartable)
MIN_RAM_GB, LIMIT_GB, LIMIT_S = (5, 9, 1800) if FULL else (1.5, 4, 600)
print("mode:", "FULL" if FULL else "TEST", "| free RAM", round(psutil.virtual_memory().available / 1e9, 1), "GB",
      "| free disk", round(shutil.disk_usage(PROJECT).free / 1e9, 1), "GB")
assert shutil.disk_usage(PROJECT).free / 1e9 > 5, "less than 5 GB free disk: stop"

mode: FULL | free RAM 6.7 GB | free disk 29.4 GB


## Benchmark inputs

In [5]:
# Inputs, written once and read by both engines: populated cells (centre lon/lat, children of primary age) and
# schools with primary classes. Kerala 10 % = cells whose hash(cell_id) % 10 = 0, with all Kerala schools.
LON, LAT = pc.lonlat_sql(PROJECT / "data" / "raw" / "worldpop")
SIZES = {"kerala_10pct": ("Kerala", "hash(cell_id) % 10 = 0"), "kerala": ("Kerala", "true"), "up": ("Uttar Pradesh", "true")}
if not FULL:
    SIZES = {"kerala_10pct": SIZES["kerala_10pct"]}
cells_glob = (SILVER / "population_cells").as_posix() + "/**/*.parquet"
schools_glob = (SILVER / "schools_india").as_posix() + "/**/*.parquet"
n_cells = {}
for size, (state, where) in SIZES.items():
    d = BENCH / size
    d.mkdir(exist_ok=True)
    if not (d / "cells.parquet").exists():
        duckdb.sql(f"""COPY (SELECT cell_id, {LON} AS lon, {LAT} AS lat, 0.8 * pop_5_9 + 0.2 * pop_10_14 AS dem
                     FROM read_parquet('{cells_glob}', hive_partitioning = true) WHERE state = '{state}' AND {where})
                     TO '{(d / "cells.parquet").as_posix()}' (FORMAT parquet)""")
        duckdb.sql(f"""COPY (SELECT row_number() OVER (ORDER BY udise_code) AS sid, lon, lat
                     FROM read_parquet('{schools_glob}', hive_partitioning = true)
                     WHERE state_name = '{state}' AND has_primary AND lat IS NOT NULL)
                     TO '{(d / "schools.parquet").as_posix()}' (FORMAT parquet)""")
    n_cells[size] = duckdb.sql(f"SELECT count(*) FROM '{(d / 'cells.parquet').as_posix()}'").fetchall()[0][0]
    n_s = duckdb.sql(f"SELECT count(*) FROM '{(d / 'schools.parquet').as_posix()}'").fetchall()[0][0]
    print(f"{size:13s} {n_cells[size]:>11,} cells x {n_s:>7,} primary schools")

kerala_10pct      296,960 cells x  10,789 primary schools
kerala          2,977,966 cells x  10,789 primary schools
up             13,754,598 cells x 119,608 primary schools


## Check: plain-Python job = brute force

In [6]:
# Check: the plain-Python job finds exactly the pairs of a brute-force search (every cell against every school,
# WGS84 ellipsoid distance) on 2,000 random cells of the Kerala sample.
import bench_job as bj
from pyproj import Geod

chk = BENCH / "check"
chk.mkdir(exist_ok=True)
c = pd.read_parquet(BENCH / "kerala_10pct" / "cells.parquet").sample(2000, random_state=1)
c.to_parquet(chk / "cells.parquet")
shutil.copy(BENCH / "kerala_10pct" / "schools.parquet", chk / "schools.parquet")
s = pd.read_parquet(chk / "schools.parquet")
geod = Geod(ellps="WGS84")
pairs, wsum = 0, 0.0
for la, lo, dem in c[["lat", "lon", "dem"]].itertuples(index=False):
    d = geod.inv(np.full(len(s), lo), np.full(len(s), la), s["lon"].to_numpy(), s["lat"].to_numpy())[2]
    k = d <= bj.D0
    pairs += int(k.sum())
    wsum += float((dem * bj.gaussian(d[k])).sum())
got = bj.python_job(chk)
assert got[0] == pairs and abs(got[1] / wsum - 1) < 1e-9, (got, pairs, wsum)
print(f"brute force on 2,000 cells: {pairs:,} pairs, weighted children {wsum:,.4f} = the KD-tree job")

brute force on 2,000 cells: 2,564 pairs, weighted children 1,406.7131 = the KD-tree job


## Run the settings

In [7]:
# Every setting runs in its own process (each core count needs a fresh Spark JVM; a process that runs out of memory
# is stopped without taking the notebook down). Memory = peak resident memory of the process and its children (JVM).
def launch(engine, size, cores, runs):
    args = [sys.executable, str(JOB), "--engine", engine, "--data", str(BENCH / size), "--cores", str(cores),
            "--runs", str(runs), "--memory", "4g" if FULL else "2g", "--project", str(PROJECT)]
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, encoding="utf-8")
    peak, status, t0 = [0.0], ["ok"], time.time()

    def watch():
        while p.poll() is None:
            try:
                procs = [psutil.Process(p.pid)] + psutil.Process(p.pid).children(recursive=True)
                rss = sum(x.memory_info().rss for x in procs) / 1e9
            except psutil.Error:
                continue
            peak[0] = max(peak[0], rss)
            if rss > LIMIT_GB or time.time() - t0 > LIMIT_S:
                status[0] = f"stopped: over {LIMIT_GB} GB" if rss > LIMIT_GB else f"stopped: over {LIMIT_S // 60} min"
                for x in psutil.Process(p.pid).children(recursive=True) + [psutil.Process(p.pid)]:
                    try:
                        x.kill()  # only this benchmark's own processes
                    except psutil.Error:
                        pass
                return
            time.sleep(0.5)

    th = threading.Thread(target=watch, daemon=True)
    th.start()
    out = p.communicate()[0]
    th.join()
    res = next((json.loads(l[7:]) for l in out.splitlines() if l.startswith("RESULT ")), None)
    if res is None and status[0] == "ok":
        status[0] = "failed (see the job's output)"
    row = {"engine": engine, "size": size, "cells": n_cells[size], "cores": cores if engine == "spark" else 1,
           "status": status[0], "peak_memory_gb": round(peak[0], 2), "wall_s": round(time.time() - t0, 1), **(res or {})}
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")
    print(f"{engine:6s} {size:13s} cores {row['cores']}: {status[0]:8s} median {row.get('median_s', '–')} s "
          f"(runs {row.get('seconds', '–')}, start-up {row.get('startup_s', '–')} s) pairs {row.get('pairs', '–')} "
          f"peak {row['peak_memory_gb']} GB")
    return row


RUNS = 3
if FULL:  # smallest first; cores on Kerala and Uttar Pradesh
    SETTINGS = [(e, s, 6) for s in SIZES for e in ("python", "spark")] + \
               [("spark", s, k) for s in ("kerala", "up") for k in (1, 2, 4)]
else:
    SETTINGS = [("python", "kerala_10pct", 1), ("spark", "kerala_10pct", 2)]
done = set()
if LOG.exists():
    for l in LOG.read_text(encoding="utf-8").splitlines():
        r = json.loads(l)
        if r["status"] == "ok":
            done.add((r["engine"], r["size"], r["cores"]))
for engine, size, cores in SETTINGS:
    key = (engine, size, cores if engine == "spark" else 1)
    if key in done:
        print("already measured:", key)
        continue
    # memory of the previous process (a closing JVM) takes a while to return: wait up to 3 min before skipping
    t_wait = time.time()
    while (free := psutil.virtual_memory().available / 1e9) < MIN_RAM_GB and time.time() - t_wait < 180:
        time.sleep(5)
    if free < MIN_RAM_GB:
        print(f"skipped {key}: only {free:.1f} GB RAM free (needs {MIN_RAM_GB})")
        continue
    launch(engine, size, cores, RUNS)

already measured: ('python', 'kerala_10pct', 1)
already measured: ('spark', 'kerala_10pct', 6)
already measured: ('python', 'kerala', 1)
already measured: ('spark', 'kerala', 6)


python up            cores 1: ok       median 45.44 s (runs [46.23, 45.44, 45.35], start-up 0.0 s) pairs 27744351 peak 2.83 GB


spark  up            cores 6: ok       median 25.56 s (runs [27.09, 24.15, 25.56], start-up 8.27 s) pairs 27744351 peak 2.78 GB


spark  kerala        cores 1: ok       median 11.41 s (runs [14.08, 11.13, 11.41], start-up 9.03 s) pairs 3712978 peak 1.51 GB


spark  kerala        cores 2: ok       median 7.36 s (runs [10.76, 7.16, 7.36], start-up 9.95 s) pairs 3712978 peak 1.7 GB


spark  kerala        cores 4: ok       median 5.71 s (runs [8.52, 5.71, 5.48], start-up 8.23 s) pairs 3712978 peak 2.28 GB


spark  up            cores 1: ok       median 82.48 s (runs [82.48, 79.84, 105.26], start-up 8.18 s) pairs 27744351 peak 2.25 GB


spark  up            cores 2: ok       median 40.4 s (runs [49.39, 40.4, 38.95], start-up 9.55 s) pairs 27744351 peak 2.29 GB


spark  up            cores 4: ok       median 24.59 s (runs [27.43, 24.59, 23.93], start-up 7.46 s) pairs 27744351 peak 2.71 GB


## Results and charts

In [8]:
# Results: same answer everywhere, then the tables and charts
import matplotlib
import matplotlib.ticker
matplotlib.use("Agg")
import matplotlib.pyplot as plt

res = pd.DataFrame([json.loads(l) for l in LOG.read_text(encoding="utf-8").splitlines()])
res = res.drop_duplicates(["engine", "size", "cores"], keep="last")
ok = res[res["status"].eq("ok")]
for size, g in ok.groupby("size"):
    assert g["pairs"].nunique() == 1, f"{size}: pair counts differ {g['pairs'].tolist()}"
    assert (g["weighted_children"] / g["weighted_children"].iloc[0] - 1).abs().max() < 1e-9, f"{size}: weighted sums differ"
print("same pairs and weighted children in every engine / core count of each size")
table = res[["size", "cells", "engine", "cores", "status", "median_s", "seconds", "startup_s", "peak_memory_gb", "pairs"]] \
    .sort_values(["cells", "engine", "cores"])
print(table.to_string(index=False))

dest = OUT if FULL else BENCH / "test_outputs"
dest.mkdir(parents=True, exist_ok=True)
table.to_csv(dest / "benchmarks.csv", index=False)
if FULL:
    table.to_csv(GOLD / "benchmarks.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 4.2))
for (eng, cores), lab, mk in [(("python", 1), "Plain Python (1 process)", "o"), (("spark", 6 if FULL else 2), f"Spark + Sedona ({6 if FULL else 2} cores)", "s")]:
    g = ok[ok["engine"].eq(eng) & ok["cores"].eq(cores)].sort_values("cells")
    ax.plot(g["cells"], g["median_s"], marker=mk, label=lab)
    for x, y in zip(g["cells"], g["median_s"]):
        ax.annotate(f"{y:.1f} s", (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xscale("log"); ax.set_yscale("log")
for axis, f in ((ax.xaxis, "{:,.0f}"), (ax.yaxis, "{:g}")):  # plain numbers, not 3x10^0
    axis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _, f=f: f.format(v)))
    axis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_xlabel("populated 100 m cells"); ax.set_ylabel("median run time (s)")
ax.set_title("Run time vs data size (primary schools, 1 km E2SFCA step)")
ax.legend(); ax.grid(alpha=.3, which="both")
fig.tight_layout(); fig.savefig(dest / "bench_size.png", dpi=150); plt.close(fig)

sp = ok[ok["engine"].eq("spark")]
if sp["cores"].nunique() > 1:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.8))
    for size, g in sp.groupby("size"):
        g = g.sort_values("cores")
        if len(g) < 2:
            continue
        a1.plot(g["cores"], g["median_s"], marker="o", label=size)
        a2.plot(g["cores"], g["median_s"].iloc[0] / g["median_s"], marker="o", label=size)
    k = sorted(sp["cores"].unique())
    a2.plot(k, [x / k[0] for x in k], "k--", lw=.8, label="ideal")
    a1.set_xlabel("cores"); a1.set_ylabel("median run time (s)"); a1.set_title("Spark run time vs cores")
    a2.set_xlabel("cores"); a2.set_ylabel(f"speed-up vs {k[0]} core"); a2.set_title("Speed-up")
    for a in (a1, a2):
        a.set_xticks(k); a.grid(alpha=.3); a.legend()
    fig.tight_layout(); fig.savefig(dest / "bench_cores.png", dpi=150); plt.close(fig)
print("written:", sorted(p.name for p in dest.iterdir()))

same pairs and weighted children in every engine / core count of each size
        size    cells engine  cores status  median_s                seconds  startup_s  peak_memory_gb    pairs
kerala_10pct   296960 python      1     ok      0.81     [1.12, 0.81, 0.78]       0.00            0.27   371045
kerala_10pct   296960  spark      6     ok      1.80       [4.2, 1.8, 1.74]       8.99            1.31   371045
      kerala  2977966 python      1     ok      7.81     [8.13, 7.81, 7.42]       0.00            0.94  3712978
      kerala  2977966  spark      1     ok     11.41  [14.08, 11.13, 11.41]       9.03            1.51  3712978
      kerala  2977966  spark      2     ok      7.36    [10.76, 7.16, 7.36]       9.95            1.70  3712978
      kerala  2977966  spark      4     ok      5.71     [8.52, 5.71, 5.48]       8.23            2.28  3712978
      kerala  2977966  spark      6     ok      4.29     [6.96, 4.29, 3.92]       7.59            2.48  3712978
          up 13754598 python 

written: ['bench_cores.png', 'bench_size.png', 'benchmarks.csv', 'five_v.csv']


## The 5 Vs

In [9]:
# The 5 Vs of this project, measured from the data lake (written with the full run)
def folder_gb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e9

w, h, _ = pc.grid(PROJECT / "data" / "raw" / "worldpop")
cells_all = duckdb.sql(f"SELECT count(*) FROM read_parquet('{cells_glob}', hive_partitioning = true)").fetchall()[0][0]
schools_all = duckdb.sql(f"SELECT count(*) FROM read_parquet('{schools_glob}', hive_partitioning = true)").fetchall()[0][0]
five_v = pd.DataFrame([
    ("Volume", f"WorldPop India grid {w:,} x {h:,} = {w * h / 1e9:.2f} bn cells; {cells_all:,} populated 100 m cells; "
               f"{schools_all:,} schools; E2SFCA result for India {folder_gb(GOLD / 'school_access_india'):.1f} GB "
               f"(about 5 h on Spark, notebook 26)"),
    ("Variety", "Rasters (WorldPop GeoTIFF), points (UDISE CSV, Sametham web pages, OSM JSON), polygons (geoBoundaries "
                "GeoJSON, OSM local bodies), PDFs (HSCAP/VHSCAP seats), stored as Parquet / GeoParquet"),
    ("Velocity", "Low: school locations and enrolment change yearly, so the analysis is batch (no streaming)"),
    ("Veracity", "107 data-quality checks (notebook 18), placeholder and outlier location fixes, coverage flags for "
                 "incomplete UDISE lists, conservation check of E2SFCA in every state"),
    ("Value", "Children beyond the distance norm and places per child for every area; inequality (Gini, Theil); "
              "hotspots, area types, priority areas; an offline decision-support dashboard"),
], columns=["V", "this project"])
print(five_v.to_string(index=False))
if FULL:
    five_v.to_csv(OUT / "five_v.csv", index=False)

       V                                                                                                                                                                                this project
  Volume                WorldPop India grid 35,040 x 34,507 = 1.21 bn cells; 140,086,378 populated 100 m cells; 1,070,017 schools; E2SFCA result for India 17.7 GB (about 5 h on Spark, notebook 26)
 Variety Rasters (WorldPop GeoTIFF), points (UDISE CSV, Sametham web pages, OSM JSON), polygons (geoBoundaries GeoJSON, OSM local bodies), PDFs (HSCAP/VHSCAP seats), stored as Parquet / GeoParquet
Velocity                                                                                                  Low: school locations and enrolment change yearly, so the analysis is batch (no streaming)
Veracity                       107 data-quality checks (notebook 18), placeholder and outlier location fixes, coverage flags for incomplete UDISE lists, conservation check of E2SFCA in every state
   Value       